<a href="https://colab.research.google.com/github/olawoye/nla_phd/blob/main/notebooks/setup_activations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# Setup my project folder and additional configuration
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Sweden/PhD/KTH_PhD_Project"

USE_GOOGLE_DRIVE = False  # Set to True to save to Drive, False to keep local

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# Lets instal basic linbraries
!pip install -q transformers
!pip install -q accelerate
!pip install -q datasets
!pip install -q pyarrow
!pip install -q pandas
!pip install -q sentencepiece

# For Gemma
!pip install -q torch

import json
import os
import torch
import pandas as pd


In [12]:
# Huggin face approval
from huggingface_hub import login

login()

In [13]:
# So I'm kind of experiementing with either Gemma-2-2B and Qwen2.5-Coder-3B (because its seems with better code inclined)
# but Gemma seems to better SAE ecosytem to leverage faster

# Gemma Option
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

model_name = "google/gemma-2-2b"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

# Should we decide on Qwen
'''
model_name = "Qwen/Qwen2.5-Coder-3B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)
'''

# Lets see
print(model)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemma2RMSNo

In [17]:
# Lets run a simple inference from the model


if USE_GOOGLE_DRIVE:
    save_dir = PROJECT_DIR + "/colab_outputs"
else:
    save_dir = "/content/local_colab_outputs"

os.makedirs(save_dir, exist_ok=True)

# --- PROMPTS FOR YOUR BULK RUN ---
prompts_list = [
    "Write a Python function that computes factorial.",
    "Write a Python function that checks prime numbers.",
    # Add your thousands of prompts here...
]

# --- manually define the Gemma chat template (our base model doesnt carry this)
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{'<start_of_turn>' + message['role'] + '\n' + message['content'] + '<end_of_turn>\n'}}"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "{{'<start_of_turn>assistant\n'}}"
        "{% endif %}"
    )

# --- BULK RUN LOOP ---
for idx, prompt_text in enumerate(prompts_list):
    chat = [{"role": "user", "content": f"{prompt_text} Output only the code."}]
    prompt = tokenizer.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Inference
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        output_hidden_states=True,
        return_dict_in_generate=True,
    )


    # --- SHAPE INSPECTION PRINTING BLOCK ---

    # 1. output.hidden_states is a tuple where each element is a generation step
    num_steps = len(output.hidden_states)

    print("Shape:\n")

    # Let's inspect the very first step (Step 0) of the generation
    first_step_layers = output.hidden_states[0]

    # Extract Layer 8 (index 8) from this step
    # This gives you the full tensor before any slicing or memory optimization
    layer_8 = first_step_layers[8]

    # Print the code and dynamic shape dynamically
    print("print(layer_8.shape)\n")
    print(f"{tuple(layer_8.shape)}\n")

    # 2. Extract dimensions from the shape tensor
    batch_size, sequence_length, hidden_dimensions = layer_8.shape

    print("Meaning:\n")
    print(f"batch size = {batch_size}\n")
    print(f"{sequence_length} tokens\n")
    print(f"{hidden_dimensions} hidden dimensions\n")
    print("-" * 40)



    # Text extraction
    input_length = inputs.input_ids.shape[1]
    generated_tokens = output.sequences[0, input_length:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    # Layer 8 Extraction (Memory Optimized)
    layer_8_activations = []
    for step in range(len(output.hidden_states)):
        # Grab layer 8, extract last token slice, drop to CPU list immediately
        last_token_act = output.hidden_states[step][8][0, -1, :]
        layer_8_activations.append(last_token_act.detach().cpu().tolist())

    # Build individual save payload
    output_data = {
        "prompt": prompt_text,
        "generated_code": generated_text,
        "layer_8_activations": layer_8_activations,
    }

    # Save incrementally to prevent data loss if Colab disconnects
    file_path = os.path.join(save_dir, f"result_{idx}.json")
    with open(file_path, "w") as f:
        json.dump(output_data, f)

print(f"Bulk run completed! Files saved in: {save_dir}")


Shape:

print(layer_8.shape)

(1, 22, 2304)

Meaning:

batch size = 1

22 tokens

2304 hidden dimensions

----------------------------------------
Shape:

print(layer_8.shape)

(1, 23, 2304)

Meaning:

batch size = 1

23 tokens

2304 hidden dimensions

----------------------------------------
Bulk run completed! Files saved in: /content/local_colab_outputs


In [18]:
# Lets get ready to do this for larger data

NUM_SAMPLES_TO_PROCESS = 100  # I can easily set to 10, 100, 10000, or 50000
CHECKPOINT_EVERY = 50        # Save data periodically to prevent loss
TARGET_LAYER = 8             # The internal transformer layer to extract according to documen
TEXT_TRUNCATION_LIMIT = 1000 # Character limit per document to keep generation fast
OUTPUT_FILENAME = "gemma_activations_final.parquet"

print(f"COnfiguring to to process {NUM_SAMPLES_TO_PROCESS} samples with checkpoints every {CHECKPOINT_EVERY} steps.")

# --- LOAD DATASET ---
from datasets import load_dataset

# Downloading/Loading OpenWebText dataset from Hugging Face (we use streaming=True download all 40GB of OpenWebText to Colab
dataset = load_dataset("Skylion007/openwebtext", split="train", streaming=True)
dataset_iter = iter(dataset)

# Test print the first item sample snippet to see it works
test_item = next(dataset_iter)
print(test_item["text"][:200] + "...")


COnfiguring to to process 100 samples with checkpoints every 50 steps.


README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Port-au-Prince, Haiti (CNN) -- Earthquake victims, writhing in pain and grasping at life, watched doctors and nurses walk away from a field hospital Friday night after a Belgian medical team evacuated...


In [19]:
# Put layer activation in a single func to re-use easily and keep tweaks in one plac

def extract_layer_activation(text_input, layer_index=8):
    """
    Passes text through the model and extracts the hidden state activation
    vector of the final token at a specific layer.
    """
    # Truncate text to keep sequence lengths uniform and fast
    truncated_text = text_input[:TEXT_TRUNCATION_LIMIT]

    inputs = tokenizer(
        truncated_text,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    # We use torch.no_grad() because we are NOT training the LLM; this saves massive memory
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # Shape: [1, seq_len, 2304]
    full_layer_tensor = outputs.hidden_states[layer_index]

    # Slicing out [0, -1] yields the final token vector -> Shape: [2304]
    final_token_vector = full_layer_tensor[0, -1, :]

    # CRITICAL STEP FOR 50K RUNS: Move to CPU and cast to numpy to free up GPU cache
    activation_array = final_token_vector.detach().cpu().numpy()

    return activation_array, truncated_text


In [21]:
# and so we start...

import pandas as pd

records = []
checkpoint_counter = 0

print(f"Starting extract loop for {NUM_SAMPLES_TO_PROCESS} iterations...")

for i in range(NUM_SAMPLES_TO_PROCESS):
    try:
        # Fetch the next document from the stream
        document = next(dataset_iter)
        raw_text = document["text"]

        # Extract activations
        activation, processed_text = extract_layer_activation(raw_text, layer_index=TARGET_LAYER)

        # Build Anthropic Recommended Dataset Structure
        records.append({
            "sample_index": i,
            "text": processed_text,
            "layer": TARGET_LAYER,
            "activation": activation.tolist() # Convert numpy array to standard list for JSON/Parquet storage
        })

    except StopIteration:
        print("Issue with dataset stream..")
        break
    except Exception as e:
        print(f"Skipped index {i} due to an error: {e}")
        continue

    # Periodic Checkpoint Saving
    if (i + 1) % CHECKPOINT_EVERY == 0:
        checkpoint_df = pd.DataFrame(records)
        checkpoint_path = f"checkpoint_step_{i + 1}.parquet"
        checkpoint_df.to_parquet(checkpoint_path)
        print(f"Saved checkpoint: {checkpoint_path} (Total recs: {len(records)})")

print(f"\nLoop completed! Total activations gathered: {len(records)}")


Starting extract loop for 100 iterations...
Saved checkpoint: checkpoint_step_50.parquet (Total recs: 50)
Saved checkpoint: checkpoint_step_100.parquet (Total recs: 100)

Loop completed! Total activations gathered: 100


In [22]:
# Add everything into a final DataFrame
final_df = pd.DataFrame(records)

# Save to file
final_df.to_parquet(OUTPUT_FILENAME)
print(f"final dataset successfully created: {OUTPUT_FILENAME}")

# Lets verify
print("\n--- DATASET VERIFICATION ---")
print(f"Total Rows: {final_df.shape[0]}")
print(f"Columns: {list(final_df.columns)}")

# Check the vector length of the first row's activation column
vector_length = len(final_df["activation"].iloc[0])
print(f"Extracted Vector Hidden Dimension Size: {vector_length} (Expected: 2304)")

# Show final layout preview
final_df.head(2)


final dataset successfully created: gemma_activations_final.parquet

--- DATASET VERIFICATION ---
Total Rows: 100
Columns: ['sample_index', 'text', 'layer', 'activation']
Extracted Vector Hidden Dimension Size: 2304 (Expected: 2304)


,sample_index,text,layer,activation
0,0,NES has signed a new contract with Remontowa S...,8,"[-0.5424960851669312, -0.05053088441491127, 1...."
1,1,"Five Labour grandees, who previously campaigne...",8,"[-0.8981407880783081, -0.24402236938476562, 0...."


In [ ]:
# Save Progress to Hithub
# 1. Configuration (Replace with your actual info)
GITHUB_TOKEN = "your_github_personal_access_token"
GITHUB_USER = "your_github_username"
REPO_NAME = "your_repository_name"

# 2. Configure Git Identity
!git config --global user.email "your_email@example.com"
!git config --global user.name "Your Name"

# 3. Clone your repository locally inside Colab
!git clone https://{GITHUB_TOKEN}@://github.com{GITHUB_USER}/{REPO_NAME}.git

# 4. Move your saved activation data files into the repo folder
# (Assuming USE_GOOGLE_DRIVE was set to False)
!cp -r /content/local_colab_outputs/* /content/{REPO_NAME}/

# 5. Commit and Push
%cd /content/{REPO_NAME}
!git add .
!git commit -m "Add Layer 8 activations from bulk run"
!git push origin main
